In [ ]:

from transformers import BertTokenizer, BertForSequenceClassification

# Load the pre-trained model (update the path as needed)
model_path = "/content/drive/MyDrive/pretrained_bert"  # Path to your pre-trained model
bert_model = BertForSequenceClassification.from_pretrained(model_path)
bert_tokenizer = BertTokenizer.from_pretrained(model_path)


# NLP Classification using BERT
This notebook demonstrates text classification using BERT with datasets in JSON format.

In [ ]:

!pip install transformers datasets


## Importing Necessary Libraries

## Dataset Paths and Preprocessing

In [ ]:

# Update these paths to point to your dataset files on Colab or Google Drive
from google.colab import drive
drive.mount('/content/drive')

TRAIN_JSON_PATH = "/content/drive/MyDrive/dataset_en_train_processed.json"
TEST_JSON_PATH = "/content/drive/MyDrive/dataset_en_test_processed.json"
PLOT_DIR = "/content/plots"

# Ensure the plot directory exists
os.makedirs(PLOT_DIR, exist_ok=True)


In [ ]:

def load_json_dataset(file_path):
    texts, labels = [], []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line.strip())  # Load each line as JSON
            texts.append(entry['processed_text'])  # Adjust with the correct key
            labels.append(entry['category'])      # Adjust with the correct key
    return texts, labels


In [ ]:

def encode_labels(labels):
    unique_labels = list(set(labels))
    label_to_index = {label: idx for idx, label in enumerate(unique_labels)}
    return [label_to_index[label] for label in labels], label_to_index


In [ ]:

# Load datasets
train_texts, train_labels = load_json_dataset(TRAIN_JSON_PATH)
test_texts, test_labels = load_json_dataset(TEST_JSON_PATH)

# Encode labels
train_labels_encoded, label_map = encode_labels(train_labels)
test_labels_encoded = [label_map[label] for label in test_labels]


## Preparing Dataset for BERT

In [ ]:

bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def prepare_dataset(texts, labels):
    encodings = bert_tokenizer(texts, truncation=True, padding=True, max_length=512)
    dataset = Dataset.from_dict({
        'input_ids': encodings['input_ids'],
        'attention_mask': encodings['attention_mask'],
        'labels': labels
    })
    return dataset

train_dataset = prepare_dataset(train_texts, train_labels_encoded)
val_dataset = prepare_dataset(test_texts, test_labels_encoded)


## Initializing and Training BERT

## Evaluating the Model

In [ ]:

predictions = trainer.predict(val_dataset)
predicted_labels = np.argmax(predictions.predictions, axis=1)

bert_f1_macro = f1_score(test_labels_encoded, predicted_labels, average='macro')
bert_f1_class_0 = f1_score(test_labels_encoded, predicted_labels, pos_label=0)
bert_f1_class_1 = f1_score(test_labels_encoded, predicted_labels, pos_label=1)
bert_mcc = matthews_corrcoef(test_labels_encoded, predicted_labels)
bert_cm = confusion_matrix(test_labels_encoded, predicted_labels)

print("BERT Results:")
print(f"F1-macro: {bert_f1_macro:.3f}")
print(f"F1-critical: {bert_f1_class_0:.3f}")
print(f"F1-conspiracy: {bert_f1_class_1:.3f}")
print(f"MCC: {bert_mcc:.3f}")
print(f"Confusion Matrix:\n{bert_cm}")

# Save results to JSON
bert_results = {
    "F1-macro": bert_f1_macro,
    "F1-critical": bert_f1_class_0,
    "F1-conspiracy": bert_f1_class_1,
    "MCC": bert_mcc,
    "Confusion Matrix": bert_cm.tolist()
}

with open(os.path.join(PLOT_DIR, "bert_results.json"), "w") as f:
    json.dump(bert_results, f, indent=4)
print(f"BERT results saved to {os.path.join(PLOT_DIR, 'bert_results.json')}")
